## Imports & Pfade

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import wfdb
from scipy.signal import resample_poly
from math import gcd


In [ ]:
DATA_ROOT = Path("../data")
RAW_ROOT = DATA_ROOT / "raw"
PROC_ROOT = DATA_ROOT / "processed"

MITDB_DIR = RAW_ROOT / "mit-bih-arrhythmia-database-1.0.0"
VFDB_DIR  = RAW_ROOT / "mit-bih-malignant-ventricular-ectopy-database-1.0.0"

OUT_DIR = PROC_ROOT / "harmonized_250hz"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("MITDB_DIR:", MITDB_DIR.resolve())
print("VFDB_DIR :", VFDB_DIR.resolve())
print("OUT_DIR  :", OUT_DIR.resolve())


## Records listen

In [ ]:
def read_records(records_file: Path):
    return [line.strip() for line in records_file.read_text().splitlines() if line.strip()]

mitdb_records = read_records(MITDB_DIR / "RECORDS")
vfdb_records  = read_records(VFDB_DIR  / "RECORDS")

print("MITDB records:", len(mitdb_records))
print("VFDB  records:", len(vfdb_records))
print("MITDB sample:", mitdb_records[:5])
print("VFDB  sample:", vfdb_records[:5])


## Resampling-funktion (360->250)

In [ ]:
def resample_to_target_fs(sig: np.ndarray, orig_fs: int, target_fs: int = 250) -> np.ndarray:
  
    if orig_fs == target_fs:
        return sig

    up = target_fs
    down = orig_fs
    g = gcd(up, down)
    up //= g
    down //= g

    sig_rs = resample_poly(sig, up, down, axis=0)
    return sig_rs


## Maligne Intervalle aus aux_note extrahieren (sekundenbasiert)

In [ ]:
MALIGNANT_LABELS = ["(VT", "(VFL", "(VFIB", "(VF"]  

def extract_malignant_intervals_seconds(db_dir: Path, record_name: str, ann_name: str = "atr"):
    rec_path = str(db_dir / record_name)
    header = wfdb.rdheader(rec_path)
    fs = header.fs

    ann = wfdb.rdann(rec_path, ann_name)

    intervals = []
    current_start = None

    for sample, note in zip(ann.sample, ann.aux_note):
        if not note:
            continue
        note = note.strip()

        if any(lbl in note for lbl in MALIGNANT_LABELS):
            current_start = sample / fs
            continue

        if current_start is not None and note.startswith("("):
            end_s = sample / fs
            if end_s > current_start:
                intervals.append((current_start, end_s))
            current_start = None

    if current_start is not None:
        end_s = header.sig_len / fs
        if end_s > current_start:
            intervals.append((current_start, end_s))

    return intervals


## NOISE-Intervalle extrahieren

In [ ]:
NOISE_LABELS = ["(NOISE"]

def extract_noise_intervals_seconds(db_dir: Path, record_name: str, ann_name: str = "atr"):
    rec_path = str(db_dir / record_name)
    header = wfdb.rdheader(rec_path)
    fs = header.fs

    ann = wfdb.rdann(rec_path, ann_name)

    intervals = []
    current_start = None

    for sample, note in zip(ann.sample, ann.aux_note):
        if not note:
            continue
        note = note.replace("\x00", "").strip()

        if any(lbl in note for lbl in NOISE_LABELS):
            current_start = sample / fs
            continue

        if current_start is not None and note.startswith("("):
            end_s = sample / fs
            if end_s > current_start:
                intervals.append((current_start, end_s))
            current_start = None

    if current_start is not None:
        end_s = header.sig_len / fs
        if end_s > current_start:
            intervals.append((current_start, end_s))

    return intervals


## Record laden + harmonisieren + speichern

In [ ]:
def process_and_save_record(db_dir: Path, source: str, record_name: str, target_fs: int = 250):
    rec_path = str(db_dir / record_name)

    rec = wfdb.rdrecord(rec_path)
    orig_fs = int(rec.fs)
    sig = rec.p_signal.astype(np.float32)  # (N, C)
    sig_name = list(rec.sig_name) if hasattr(rec, "sig_name") else None

    intervals_s = extract_malignant_intervals_seconds(db_dir, record_name, ann_name="atr")
    intervals_s = np.array(intervals_s, dtype=np.float32).reshape(-1, 2)

    noise_s = extract_noise_intervals_seconds(db_dir, record_name, ann_name="atr")
    noise_s = np.array(noise_s, dtype=np.float32).reshape(-1, 2)

    sig_rs = resample_to_target_fs(sig, orig_fs=orig_fs, target_fs=target_fs).astype(np.float32)

    out_path = OUT_DIR / f"{source}_{record_name}.npz"
    np.savez_compressed(
        out_path,
        signal=sig_rs,
        fs=np.array(target_fs, dtype=np.int32),
        source=np.array(source),
        record=np.array(record_name),
        sig_name=np.array(sig_name, dtype=object),
        orig_fs=np.array(orig_fs, dtype=np.int32),
        malignant_intervals_s=intervals_s,
        noise_intervals_s=noise_s
    )

    noise_total_s = float(noise_s[:, 1].sum() - noise_s[:, 0].sum()) if noise_s.size else 0.0

    return {
        "source": source,
        "record": record_name,
        "orig_fs": orig_fs,
        "target_fs": target_fs,
        "orig_len": sig.shape[0],
        "rs_len": sig_rs.shape[0],
        "channels": sig.shape[1],
        "n_malignant_intervals": int(intervals_s.shape[0]),
        "malignant_total_s": float(intervals_s[:,1].sum() - intervals_s[:,0].sum()) if intervals_s.size else 0.0,
        "n_noise_intervals": int(noise_s.shape[0]),
        "noise_total_s": noise_total_s,
        "out_file": out_path.name
    }


## MITDB harmonisieren

In [ ]:
mitdb_stats = []
for r in mitdb_records:
    s = process_and_save_record(MITDB_DIR, source="mitdb", record_name=r, target_fs=250)
    mitdb_stats.append(s)

mitdb_df = pd.DataFrame(mitdb_stats)
mitdb_df.sort_values(["n_malignant_intervals", "malignant_total_s"], ascending=False).head(10)


## VFDB harmonisieren

In [ ]:
vfdb_stats = []
for r in vfdb_records:
    s = process_and_save_record(VFDB_DIR, source="vfdb", record_name=r, target_fs=250)
    vfdb_stats.append(s)

vfdb_df = pd.DataFrame(vfdb_stats)
vfdb_df.sort_values(["malignant_total_s", "n_malignant_intervals"], ascending=False).head(10)
